# 01 - Introducao ao ARIMA(p,d,q)

Este notebook apresenta os fundamentos do modelo ARIMA usando a biblioteca **chronobox**.

**Conteudo:**
1. Teoria do modelo ARIMA(p,d,q)
2. Identificacao de ordem via ACF/PACF
3. Ajuste no dataset Nile
4. Diagnostico de residuos (Ljung-Box)
5. Previsao com intervalos de confianca
6. Exercicios

## 1. Fundamentacao Teorica

O modelo **ARIMA(p, d, q)** combina tres componentes:

- **AR(p)** - Autoregressivo: a serie depende de seus proprios valores passados
  $$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \varepsilon_t$$

- **I(d)** - Integrado: a serie precisa de *d* diferenciacoes para se tornar estacionaria
  $$\Delta^d y_t = (1 - L)^d y_t$$

- **MA(q)** - Media Movel: a serie depende de erros passados
  $$y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$$

O modelo completo em notacao de operador de defasagem:

$$\phi(L)(1-L)^d y_t = c + \theta(L) \varepsilon_t$$

onde $\varepsilon_t \sim WN(0, \sigma^2)$.

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import ARIMA
from chronobox.tests_stat import adf_test, kpss_test, ljung_box_test, jarque_bera_test
from chronobox.visualization import plot_diagnostics, plot_forecast, plot_series, set_theme

set_theme('professional')
np.random.seed(42)

print('chronobox importado com sucesso!')

In [ ]:
# Carregar dataset Nile (vazao anual do Rio Nilo em Aswan, 1871-1970)
import os
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

nile = pd.read_csv(os.path.join(DATA_DIR, 'nile.csv'), parse_dates=['date'])
nile.set_index('date', inplace=True)

y = nile['flow'].values

print(f'Periodo: {nile.index[0].year} - {nile.index[-1].year}')
print(f'Observacoes: {len(y)}')
print(f'Media: {y.mean():.1f}, Desvio: {y.std():.1f}')
nile.head()

## 3. Analise Exploratoria e Estacionariedade

In [ ]:
# Grafico 1: Serie temporal do Nile
fig = plot_series(y, labels=['Vazao do Nilo'], title='Vazao Anual do Rio Nilo (1871-1970)')
plt.xlabel('Observacao')
plt.ylabel('Vazao (10^8 m^3)')
plt.show()

In [ ]:
# Teste ADF - H0: raiz unitaria (serie nao estacionaria)
adf_result = adf_test(y, regression='c')
print('=== Teste ADF ===')
print(f'Estatistica: {adf_result.statistic:.4f}')
print(f'P-valor: {adf_result.pvalue:.4f}')
print(f'Valores criticos: {adf_result.critical_values}')
print(f'Conclusao: {"Estacionaria" if adf_result.pvalue < 0.05 else "Nao estacionaria (d >= 1)"}')

print()

# Teste KPSS - H0: serie estacionaria
kpss_result = kpss_test(y, regression='c')
print('=== Teste KPSS ===')
print(f'Estatistica: {kpss_result.statistic:.4f}')
print(f'P-valor: {kpss_result.pvalue:.4f}')
print(f'Conclusao: {"Estacionaria" if kpss_result.pvalue > 0.05 else "Nao estacionaria"}')

In [ ]:
# Primeira diferenca
y_diff = np.diff(y)

adf_diff = adf_test(y_diff, regression='c')
print(f'ADF na serie diferenciada: estatistica={adf_diff.statistic:.4f}, p-valor={adf_diff.pvalue:.4f}')
print(f'Conclusao: d=1 e suficiente para estacionariedade' if adf_diff.pvalue < 0.05 else 'Precisa mais diferenciacoes')

## 4. Identificacao via ACF/PACF

Regras praticas para identificacao da ordem:

| Padrao | ACF | PACF | Modelo |
|--------|-----|------|--------|
| AR(p) | Decai exponencialmente | Corta apos lag p | ARIMA(p,d,0) |
| MA(q) | Corta apos lag q | Decai exponencialmente | ARIMA(0,d,q) |
| ARMA(p,q) | Decai apos lag q | Decai apos lag p | ARIMA(p,d,q) |

In [ ]:
# Grafico 2: ACF e PACF da serie diferenciada
from chronobox.visualization.diagnostics_plot import _compute_acf, _compute_pacf

max_lag = 20
acf_vals = _compute_acf(y_diff, max_lag)
pacf_vals = _compute_pacf(y_diff, max_lag)
ci = 1.96 / np.sqrt(len(y_diff))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(max_lag + 1), acf_vals, width=0.3, color='steelblue')
axes[0].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[0].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('ACF - Serie Diferenciada')
axes[0].set_xlabel('Lag')

axes[1].bar(range(max_lag + 1), pacf_vals, width=0.3, color='darkorange')
axes[1].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('PACF - Serie Diferenciada')
axes[1].set_xlabel('Lag')

plt.tight_layout()
plt.show()

print('Interpretacao: ACF com decaimento lento sugere componente AR.')
print('PACF cortando apos lag 1-2 sugere AR(1) ou AR(2).')
print('Modelo candidato: ARIMA(1,1,1)')

## 5. Ajuste do Modelo ARIMA com chronobox

In [ ]:
# Ajustar ARIMA(1,1,1) no dataset Nile
model = ARIMA(order=(1, 1, 1))
results = model.fit(y)

print(results.summary())

In [ ]:
# Comparacao de modelos candidatos
modelos = {
    'ARIMA(1,1,0)': ARIMA(order=(1, 1, 0)),
    'ARIMA(0,1,1)': ARIMA(order=(0, 1, 1)),
    'ARIMA(1,1,1)': ARIMA(order=(1, 1, 1)),
    'ARIMA(2,1,1)': ARIMA(order=(2, 1, 1)),
}

print(f'{"Modelo":<18} {"AIC":>10} {"BIC":>10} {"Log-Lik":>10}')
print('-' * 50)
resultados = {}
for nome, mod in modelos.items():
    res = mod.fit(y)
    resultados[nome] = res
    print(f'{nome:<18} {res.aic:>10.2f} {res.bic:>10.2f} {res.loglike:>10.2f}')

melhor = min(resultados, key=lambda k: resultados[k].aic)
print(f'\nMelhor modelo por AIC: {melhor}')

## 6. Diagnostico de Residuos

In [ ]:
# Grafico 3: Diagnostico completo dos residuos (2x2)
fig = plot_diagnostics(results, lags=15, title='Diagnostico - ARIMA(1,1,1) Nile')
plt.show()

In [ ]:
# Teste formal de Ljung-Box para autocorrelacao nos residuos
# H0: residuos sao ruido branco (nao ha autocorrelacao)
resid = results.residuals
p_order = 1  # p + q do modelo

for lag in [5, 10, 15, 20]:
    lb = ljung_box_test(resid, lags=lag, model_df=p_order)
    status = 'OK (ruido branco)' if lb.pvalue > 0.05 else 'FALHA (autocorrelacao)'
    print(f'Ljung-Box(lag={lag:2d}): Q={lb.statistic:8.3f}, p-valor={lb.pvalue:.4f} -> {status}')

print()

# Teste de normalidade Jarque-Bera
jb = jarque_bera_test(resid)
print(f'Jarque-Bera: JB={jb.statistic:.3f}, p-valor={jb.pvalue:.4f}')
print(f'Normalidade: {"Nao rejeitada" if jb.pvalue > 0.05 else "Rejeitada"}')

## 7. Previsao com Intervalos de Confianca

In [ ]:
# Previsao 10 passos a frente
fc = results.forecast(steps=10, alpha=0.05)

print('Previsoes com IC 95%:')
print(f'{"Passo":>6} {"Previsao":>12} {"IC Inferior":>12} {"IC Superior":>12}')
print('-' * 45)
for i in range(len(fc['forecast'])):
    print(f'{i+1:>6} {fc["forecast"][i]:>12.1f} {fc["lower"][i]:>12.1f} {fc["upper"][i]:>12.1f}')

In [ ]:
# Grafico 4: Previsao com fan chart
fig = plot_forecast(results, steps=10, alpha=0.05,
                    title='Previsao ARIMA(1,1,1) - Vazao do Nilo')
plt.show()

In [ ]:
# Grafico 5: Valores ajustados vs observados
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(results.fitted_values, label='Ajustado', color='darkorange', linewidth=1.2, linestyle='--')
ax.set_title('ARIMA(1,1,1): Observado vs Ajustado')
ax.set_xlabel('Observacao')
ax.set_ylabel('Vazao')
ax.legend()
plt.tight_layout()
plt.show()

## Exercicio

### Exercicio 1: Ajustar ARIMA no dataset airline.csv

1. Carregue o dataset `airline.csv` do diretorio `../data/`
2. Plote a serie temporal e identifique tendencia e sazonalidade
3. Aplique diferenciacoes necessarias e analise ACF/PACF
4. Ajuste um ARIMA (ignorando sazonalidade por enquanto) e avalie os residuos
5. Compare pelo menos 3 ordens diferentes usando AIC

**Dica:** O airline.csv possui sazonalidade forte - um ARIMA puro nao captura bem esse padrao. No proximo notebook veremos como o SARIMA resolve isso.

In [ ]:
# Exercicio 1 - Seu codigo aqui
# airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
# ...

## Exercicio

### Exercicio 2: Impacto da ordem de diferenciacao

1. Ajuste ARIMA(1,0,1), ARIMA(1,1,1) e ARIMA(1,2,1) no Nile
2. Compare AIC e BIC de cada modelo
3. Plote as previsoes de cada modelo lado a lado
4. O que acontece com a variancia da previsao quando d aumenta?

In [ ]:
# Exercicio 2 - Seu codigo aqui
# ...